# Classical ML vs Transfer Learning for Rice Disease Detection
## An Empirical Study on Paddy Doctor

**Three-tier experimental hierarchy:**

| Tier | Methods | Features |
|---|---|---|
| 1 — Classical ML | SVM, Random Forest, XGBoost, LightGBM | HOG, LBP, Color Histogram |
| 2 — Deep Learning | ResNet34, MobileNetV3, EfficientNetV2-S | End-to-end fine-tuning |
| 3 — Hybrid (novel) | SVM, XGBoost on CNN features | ResNet34 backbone → 512-D embeddings |

**Research question:**
> Can classical ML classifiers compete with end-to-end deep learning  
> when supplied with deep visual features from a pretrained CNN backbone?

**Dataset:** Paddy Doctor — 16,225 images · 13 classes  
**Estimated runtime:** 60–90 min on Colab T4 GPU

---

## 0 · Setup

In [ ]:
# Check GPU first
import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip())

!pip install -q kaggle xgboost lightgbm torchmetrics scikit-learn
print('Packages ready.')

In [ ]:
import os, json, time, copy, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# CV / ML
import cv2
from skimage.feature import hog, local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import xgboost as xgb
import lightgbm as lgb

# DL
import torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as T
import torchvision.models as tvm
from torchmetrics import F1Score

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS  = Path('/content/results'); RESULTS.mkdir(exist_ok=True)
DATA_DIR = Path('/content/paddy-disease-classification')
IMG_SIZE = 224
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 1 · Download Paddy Doctor

In [ ]:
from google.colab import files
print('Upload kaggle.json (Kaggle → Account → Create API Token):')
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c paddy-disease-classification -p /content/ -q
!unzip -q /content/paddy-disease-classification.zip -d /content/paddy-disease-classification

TRAIN_DIR = DATA_DIR / 'train_images'
EXTS = {'.jpg','.jpeg','.png'}

CLASS_NAMES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
C2I = {c: i for i, c in enumerate(CLASS_NAMES)}

print(f'\nClasses ({NUM_CLASSES}):')
total = 0
for c in CLASS_NAMES:
    n = len([f for f in (TRAIN_DIR/c).iterdir() if f.suffix in EXTS])
    total += n
    print(f'  {c:<35} {n:>5}')
print(f'Total: {total}')

In [ ]:
# ── Collect all paths + labels ────────────────────────────────────
all_paths, all_labels = [], []
for c in CLASS_NAMES:
    for p in (TRAIN_DIR/c).iterdir():
        if p.suffix in EXTS:
            all_paths.append(str(p))
            all_labels.append(C2I[c])

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

# Stratified 80/20 split — same split used for ALL tiers
tr_paths, va_paths, tr_labels, va_labels = train_test_split(
    all_paths, all_labels, test_size=0.2,
    stratify=all_labels, random_state=SEED
)
print(f'Train: {len(tr_paths)}  |  Val: {len(va_paths)}')

## Tier 1 · Classical ML

**Feature extraction pipeline:**  
`Image → resize 128×128 → HOG / LBP / Color Histogram → concatenate → PCA → SVM / RF / XGBoost / LightGBM`

128px chosen (not 224) — faster extraction, negligible quality loss for hand-crafted features.

In [ ]:
# ── Feature extractors ────────────────────────────────────────────
ML_SIZE = 128   # resize target for classical features

def extract_hog(img_bgr):
    """HOG on grayscale, 128×128."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    feat, _ = hog(gray, orientations=9, pixels_per_cell=(16,16),
                  cells_per_block=(2,2), visualize=True, feature_vector=True)
    return feat


def extract_lbp(img_bgr):
    """LBP histogram, radius=3, 24 points."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lbp  = local_binary_pattern(gray, P=24, R=3, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=26, range=(0,26), density=True)
    return hist


def extract_color_hist(img_bgr):
    """HSV color histogram (32 bins per channel)."""
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    feat = []
    for ch in range(3):
        h, _ = np.histogram(hsv[:,:,ch].ravel(), bins=32, density=True)
        feat.extend(h)
    return np.array(feat)


def extract_all(path):
    """Concatenate HOG + LBP + Color Histogram for one image."""
    img = cv2.imread(str(path))
    img = cv2.resize(img, (ML_SIZE, ML_SIZE))
    return np.concatenate([extract_hog(img),
                           extract_lbp(img),
                           extract_color_hist(img)])


# Test on one image
sample = extract_all(tr_paths[0])
print(f'Feature vector length: {len(sample)}')
hog_len  = len(extract_hog(cv2.resize(cv2.imread(tr_paths[0]),(ML_SIZE,ML_SIZE))))
lbp_len  = 26
col_len  = 96
print(f'  HOG: {hog_len}  LBP: {lbp_len}  Color hist: {col_len}')

In [ ]:
# ── Extract features for all images (CPU, ~5 min) ─────────────────
print('Extracting hand-crafted features...')
t0 = time.time()

X_train_ml = np.array([extract_all(p) for p in tqdm(tr_paths, desc='Train')])
X_val_ml   = np.array([extract_all(p) for p in tqdm(va_paths, desc='Val')])
y_train_ml = tr_labels
y_val_ml   = va_labels

print(f'Extraction done in {(time.time()-t0)/60:.1f} min')
print(f'X_train: {X_train_ml.shape}  |  X_val: {X_val_ml.shape}')

# Save — no need to re-extract if Colab disconnects
np.save(RESULTS/'X_train_ml.npy', X_train_ml)
np.save(RESULTS/'X_val_ml.npy',   X_val_ml)
np.save(RESULTS/'y_train_ml.npy', y_train_ml)
np.save(RESULTS/'y_val_ml.npy',   y_val_ml)
print('Features saved to /content/results/')

In [ ]:
# ── Scaling + PCA (keeps SVM tractable) ──────────────────────────
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train_ml)
X_va_s = scaler.transform(X_val_ml)

# PCA: 200 components ≈ covers >95% variance, speeds up SVM significantly
pca = PCA(n_components=200, random_state=SEED)
X_tr_pca = pca.fit_transform(X_tr_s)
X_va_pca = pca.transform(X_va_s)
print(f'Explained variance with 200 PCs: {pca.explained_variance_ratio_.sum():.3f}')

In [ ]:
# ── Helper: evaluate any sklearn model ───────────────────────────
ml_results = {}   # stores all results

def eval_ml(name, model, X_tr, y_tr, X_va, y_va, use_pca=True):
    Xtr = X_tr_pca if use_pca else X_tr_s
    Xva = X_va_pca if use_pca else X_va_s

    t0 = time.time()
    model.fit(Xtr, y_tr)
    train_time = time.time() - t0

    t0 = time.time()
    preds = model.predict(Xva)
    infer_time_ms = (time.time() - t0) / len(y_va) * 1000

    acc = accuracy_score(y_va, preds) * 100
    f1  = f1_score(y_va, preds, average='macro') * 100

    ml_results[name] = dict(acc=round(acc,2), f1=round(f1,2),
                             train_s=round(train_time,1),
                             infer_ms=round(infer_time_ms,3),
                             params='N/A', preds=preds)
    print(f'[{name}]  Acc={acc:.2f}%  F1={f1:.2f}%  '
          f'TrainTime={train_time:.1f}s  InferTime={infer_time_ms:.3f}ms/img')
    return preds

print('ML evaluation helper ready.')

In [ ]:
# ── 1. SVM (RBF kernel) ───────────────────────────────────────────
# C=10 and gamma='scale' are strong defaults; no grid search needed for baseline
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED)
eval_ml('HOG+LBP+Color → SVM', svm, X_tr_s, y_train_ml, X_va_s, y_val_ml)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_val_ml, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── 2. Random Forest ─────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=300, max_depth=None,
                             n_jobs=-1, random_state=SEED)
eval_ml('HOG+LBP+Color → RF', rf, X_tr_s, y_train_ml, X_va_s, y_val_ml,
        use_pca=False)  # RF handles raw features well without PCA

In [ ]:
# ── 3. XGBoost ───────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    tree_method='hist', device='cuda',   # GPU if available
    random_state=SEED, verbosity=0
)
eval_ml('HOG+LBP+Color → XGBoost', xgb_model,
        X_tr_s, y_train_ml, X_va_s, y_val_ml, use_pca=False)

In [ ]:
# ── 4. LightGBM ──────────────────────────────────────────────────
lgbm = lgb.LGBMClassifier(
    n_estimators=300, num_leaves=63, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=SEED, verbose=-1
)
eval_ml('HOG+LBP+Color → LightGBM', lgbm,
        X_tr_s, y_train_ml, X_va_s, y_val_ml, use_pca=False)

## Tier 2 · Deep Learning Baselines

Standard ImageNet fine-tuning: freeze nothing, train end-to-end.  
20 epochs each — enough for convergence on 13K images.

In [ ]:
# ── PyTorch Dataset ───────────────────────────────────────────────
class PaddyDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.transform(img), int(self.labels[i])

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6,1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.ColorJitter(0.3,0.3,0.3,0.1),
    T.ToTensor(), T.Normalize(MEAN,STD)
])
val_tf = T.Compose([
    T.Resize(256), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(MEAN,STD)
])

dl_tr = DataLoader(PaddyDataset(tr_paths, tr_labels, train_tf),
                   batch_size=64, shuffle=True, num_workers=4,
                   persistent_workers=True, pin_memory=True)
dl_va = DataLoader(PaddyDataset(va_paths, va_labels, val_tf),
                   batch_size=64, shuffle=False, num_workers=4,
                   persistent_workers=True, pin_memory=True)
print(f'Train batches: {len(dl_tr)} | Val batches: {len(dl_va)}')

In [ ]:
import timm

dl_results = {}   # stores DL results

def build_dl_model(name):
    if name == 'resnet34':
        m = tvm.resnet34(weights='IMAGENET1K_V1')
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        params = sum(p.numel() for p in m.parameters()) / 1e6
    elif name == 'mobilenetv3':
        m = tvm.mobilenet_v3_small(weights='IMAGENET1K_V1')
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
        params = sum(p.numel() for p in m.parameters()) / 1e6
    elif name == 'efficientnetv2':
        m = timm.create_model('efficientnetv2_s', pretrained=True, num_classes=NUM_CLASSES)
        params = sum(p.numel() for p in m.parameters()) / 1e6
    return m.to(DEVICE), round(params, 1)


def train_dl(name, model, epochs=20):
    crit  = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt   = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scl   = torch.cuda.amp.GradScaler()
    best_f1, best_w, hist = 0.0, None, []

    for ep in range(1, epochs+1):
        model.train()
        t0 = time.time()
        tr_correct = tr_total = 0
        for imgs, lbls in dl_tr:
            imgs, lbls = imgs.to(DEVICE,non_blocking=True), lbls.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                out  = model(imgs)
                loss = crit(out, lbls)
            scl.scale(loss).backward()
            scl.step(opt); scl.update()
            tr_correct += (out.argmax(1)==lbls).sum().item()
            tr_total   += lbls.size(0)
        sched.step()

        # Eval
        model.eval()
        preds_all, lbls_all = [], []
        with torch.no_grad():
            for imgs, lbls in dl_va:
                preds_all.append(model(imgs.to(DEVICE)).argmax(1).cpu())
                lbls_all.append(lbls)
        P = torch.cat(preds_all); L = torch.cat(lbls_all)
        val_acc = (P==L).float().mean().item()
        val_f1  = F1Score(task='multiclass',num_classes=NUM_CLASSES,average='macro')(P,L).item()
        hist.append(dict(ep=ep, tr_acc=tr_correct/tr_total,
                         val_acc=val_acc, val_f1=val_f1))
        if val_f1 > best_f1: best_f1, best_w = val_f1, copy.deepcopy(model.state_dict())
        print(f'[{name}] ep{ep:02d}/{epochs} '
              f'tr={tr_correct/tr_total:.3f} val_acc={val_acc:.3f} '
              f'val_f1={val_f1:.3f} ({time.time()-t0:.0f}s)')

    model.load_state_dict(best_w)
    return model, hist, best_f1, P, L

print('DL utilities ready.')

In [ ]:
# ── Train DL models ───────────────────────────────────────────────
DL_MODELS = ['resnet34', 'mobilenetv3', 'efficientnetv2']
DL_EPOCHS = 20

for arch in DL_MODELS:
    print(f'\n{"="*60}\nTraining: {arch}\n{"="*60}')
    random.seed(SEED); torch.manual_seed(SEED)
    model, n_params = build_dl_model(arch)

    # Measure latency (before training — architecture property)
    model.eval()
    dummy = torch.randn(32,3,224,224).to(DEVICE)
    with torch.no_grad():
        for _ in range(5): model(dummy)   # warm-up
        t0 = time.time()
        for _ in range(20): model(dummy)
        lat_ms = (time.time()-t0)/20/32*1000

    t_total = time.time()
    model, hist, best_f1, preds, labels = train_dl(arch, model, epochs=DL_EPOCHS)
    train_s = time.time() - t_total

    val_acc = accuracy_score(labels.numpy(), preds.numpy()) * 100
    val_f1  = f1_score(labels.numpy(), preds.numpy(), average='macro') * 100

    dl_results[arch] = dict(
        acc=round(val_acc,2), f1=round(val_f1,2),
        train_s=round(train_s,1), infer_ms=round(lat_ms,3),
        params=n_params, preds=preds.numpy(), labels=labels.numpy(),
        history=hist
    )
    torch.save(model.state_dict(), RESULTS/f'{arch}.pth')
    print(f'\n  → Acc={val_acc:.2f}%  F1={val_f1:.2f}%  '
          f'Params={n_params}M  Lat={lat_ms:.3f}ms/img')

## Tier 3 · Hybrid: CNN Feature Extractor + Classical ML

The novel contribution of this paper.

```
Rice Image → ResNet34 backbone (frozen) → 512-D embedding → SVM / XGBoost
```

Research question: *Can classical heads match end-to-end DL when given deep features?*  
If yes → classical heads are interpretable, faster, and deployment-friendly.

In [ ]:
# ── Extract ResNet34 embeddings (512-D) for all images ───────────
print('Loading ResNet34 backbone (ImageNet pretrained)...')
extractor = tvm.resnet34(weights='IMAGENET1K_V1')
extractor.fc = nn.Identity()   # remove classifier head → 512-D output
extractor = extractor.to(DEVICE).eval()

@torch.no_grad()
def extract_embeddings(paths, labels, bs=128):
    ds  = PaddyDataset(paths, labels, val_tf)
    ldr = DataLoader(ds, batch_size=bs, shuffle=False,
                     num_workers=4, pin_memory=True)
    feats, labs = [], []
    for imgs, lbls in tqdm(ldr, desc='Extracting CNN embeddings'):
        with torch.cuda.amp.autocast():
            f = extractor(imgs.to(DEVICE)).cpu()
        feats.append(f); labs.append(lbls)
    return torch.cat(feats).numpy(), torch.cat(labs).numpy()

t0 = time.time()
X_tr_emb, y_tr_emb = extract_embeddings(tr_paths, tr_labels)
X_va_emb, y_va_emb = extract_embeddings(va_paths, va_labels)
print(f'Embeddings extracted in {time.time()-t0:.0f}s')
print(f'Shape: train {X_tr_emb.shape}  val {X_va_emb.shape}')

np.save(RESULTS/'X_tr_emb.npy', X_tr_emb)
np.save(RESULTS/'X_va_emb.npy', X_va_emb)

In [ ]:
# ── Hybrid evaluator ─────────────────────────────────────────────
hyb_results = {}

def eval_hybrid(name, model):
    # Scale embeddings
    sc  = StandardScaler()
    Xtr = sc.fit_transform(X_tr_emb)
    Xva = sc.transform(X_va_emb)

    t0 = time.time()
    model.fit(Xtr, y_tr_emb)
    train_s = time.time() - t0

    t0 = time.time()
    preds = model.predict(Xva)
    infer_ms = (time.time()-t0)/len(y_va_emb)*1000

    acc = accuracy_score(y_va_emb, preds)*100
    f1  = f1_score(y_va_emb, preds, average='macro')*100

    hyb_results[name] = dict(acc=round(acc,2), f1=round(f1,2),
                              train_s=round(train_s,1),
                              infer_ms=round(infer_ms,3),
                              params='N/A (+21.3M backbone)',
                              preds=preds)
    print(f'[{name}]  Acc={acc:.2f}%  F1={f1:.2f}%  '
          f'TrainTime={train_s:.1f}s  InferTime={infer_ms:.3f}ms/img')

print('Hybrid evaluator ready.')

In [ ]:
# ── 1. CNN (ResNet34) + SVM ───────────────────────────────────────
eval_hybrid('ResNet34 → SVM',
            SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED))

In [ ]:
# ── 2. CNN (ResNet34) + XGBoost ───────────────────────────────────
eval_hybrid('ResNet34 → XGBoost',
            xgb.XGBClassifier(
                n_estimators=300, max_depth=6, learning_rate=0.1,
                tree_method='hist', device='cuda',
                use_label_encoder=False, eval_metric='mlogloss',
                random_state=SEED, verbosity=0
            ))

In [ ]:
# ── 3. CNN (ResNet34) + LightGBM ──────────────────────────────────
eval_hybrid('ResNet34 → LightGBM',
            lgb.LGBMClassifier(
                n_estimators=300, num_leaves=63, learning_rate=0.1,
                n_jobs=-1, random_state=SEED, verbose=-1
            ))

In [ ]:
# ── 4. CNN (ResNet34) + Random Forest ─────────────────────────────
eval_hybrid('ResNet34 → RF',
            RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED))

## Results & Analysis

In [ ]:
# ── Master results table ──────────────────────────────────────────
def make_row(name, tier, r):
    return {'Method': name, 'Tier': tier,
            'Acc (%)': r['acc'], 'F1 (%)': r['f1'],
            'Train time (s)': r['train_s'],
            'Infer (ms/img)': r['infer_ms'],
            'Params (M)': r.get('params','N/A')}

rows = []
for name, r in ml_results.items():
    rows.append(make_row(name, 'Classical ML', r))
for name, r in dl_results.items():
    rows.append(make_row(name, 'Deep Learning', r))
for name, r in hyb_results.items():
    rows.append(make_row(name, 'Hybrid', r))

master_df = pd.DataFrame(rows)
master_df = master_df.sort_values('F1 (%)', ascending=False).reset_index(drop=True)
master_df.to_csv(RESULTS/'master_results.csv', index=False)

print('=== MASTER RESULTS TABLE (sorted by F1) ===')
print(master_df.to_string(index=False))

In [ ]:
# ── 6-panel figure ────────────────────────────────────────────────
TIER_COLORS = {'Classical ML':'#FF9800', 'Deep Learning':'#2196F3', 'Hybrid':'#4CAF50'}
bar_colors  = [TIER_COLORS[t] for t in master_df['Tier']]
methods     = master_df['Method'].tolist()
short_names = [m.replace('HOG+LBP+Color → ','').replace('ResNet34 → ','CNN→') for m in methods]

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# 1 · F1 comparison
ax = fig.add_subplot(gs[0, :])
bars = ax.bar(short_names, master_df['F1 (%)'], color=bar_colors,
              edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
ax.set_title('Val Macro-F1 by Method and Tier', fontweight='bold', fontsize=13)
ax.set_ylabel('Macro-F1 (%)')
ax.set_ylim(0, 110)
ax.tick_params(axis='x', rotation=30)
for tick, lbl in zip(ax.get_xticklabels(), master_df['Tier']):
    tick.set_color(TIER_COLORS[lbl])
# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=v, label=k) for k,v in TIER_COLORS.items()],
          loc='lower right', fontsize=10)
ax.grid(True, alpha=0.2, axis='y')

# 2 · Accuracy comparison
ax2 = fig.add_subplot(gs[1, 0])
ax2.barh(short_names, master_df['Acc (%)'], color=bar_colors, edgecolor='white')
ax2.set_title('Accuracy (%)', fontweight='bold')
ax2.set_xlabel('Accuracy (%)'); ax2.set_xlim(0, 105)
ax2.grid(True, alpha=0.2, axis='x')

# 3 · Training time
ax3 = fig.add_subplot(gs[1, 1])
ax3.barh(short_names, master_df['Train time (s)'], color=bar_colors, edgecolor='white')
ax3.set_title('Training Time (seconds)', fontweight='bold')
ax3.set_xlabel('Seconds'); ax3.set_xscale('log')
ax3.grid(True, alpha=0.2, axis='x')

# 4 · Inference time
ax4 = fig.add_subplot(gs[1, 2])
ax4.barh(short_names, master_df['Infer (ms/img)'], color=bar_colors, edgecolor='white')
ax4.set_title('Inference Time (ms/image)', fontweight='bold')
ax4.set_xlabel('ms / image')
ax4.grid(True, alpha=0.2, axis='x')

fig.suptitle(
    'Classical ML vs Transfer Learning for Rice Disease Detection\n'
    'Paddy Doctor Dataset · Empirical Study',
    fontsize=14, fontweight='bold'
)
plt.savefig(RESULTS/'main_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# ── DL training curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, len(DL_MODELS), figsize=(16, 5), sharey=True)
for ax, arch in zip(axes, DL_MODELS):
    hist = dl_results[arch]['history']
    eps  = [h['ep'] for h in hist]
    ax.plot(eps, [h['tr_acc']*100  for h in hist], 'b-', label='Train Acc', lw=2)
    ax.plot(eps, [h['val_acc']*100 for h in hist], 'r-', label='Val Acc',   lw=2)
    ax.plot(eps, [h['val_f1']*100  for h in hist], 'g--',label='Val F1',    lw=2)
    ax.set_title(arch, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Score (%)')
fig.suptitle('Deep Learning Training Curves', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS/'dl_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrix — best model overall ────────────────────────
best_row  = master_df.iloc[0]
best_name = best_row['Method']
best_tier = best_row['Tier']

if best_tier == 'Classical ML':
    preds_best = ml_results[best_name]['preds']
    true_best  = y_val_ml
elif best_tier == 'Deep Learning':
    # arch key lookup
    preds_best = dl_results[best_name]['preds']
    true_best  = dl_results[best_name]['labels']
else:
    preds_best = hyb_results[best_name]['preds']
    true_best  = y_va_emb

cm   = confusion_matrix(true_best, preds_best)
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.3, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS/'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClassification report — {best_name}:')
print(classification_report(true_best, preds_best,
                             target_names=CLASS_NAMES, digits=3, zero_division=0))

In [ ]:
# ── Radar chart: multi-dimension comparison ───────────────────────
# Normalize each metric 0-1 for radar
radar_methods = list(ml_results.keys()) + list(dl_results.keys()) + list(hyb_results.keys())
f1s    = [ml_results.get(m, dl_results.get(m, hyb_results.get(m,{})))['f1'] for m in radar_methods]
accs   = [ml_results.get(m, dl_results.get(m, hyb_results.get(m,{})))['acc'] for m in radar_methods]
speeds = [1/(ml_results.get(m, dl_results.get(m, hyb_results.get(m,{})))['infer_ms']+1e-9) for m in radar_methods]

# F1 vs Accuracy scatter with tier coloring
fig, ax = plt.subplots(figsize=(10, 7))
for name, r in {**ml_results}.items():
    ax.scatter(r['acc'], r['f1'], c=TIER_COLORS['Classical ML'],
               s=120, zorder=5, marker='o')
    ax.annotate(name.replace('HOG+LBP+Color → ',''), (r['acc'], r['f1']),
                fontsize=8, xytext=(4,4), textcoords='offset points')
for name, r in {**dl_results}.items():
    ax.scatter(r['acc'], r['f1'], c=TIER_COLORS['Deep Learning'],
               s=180, zorder=5, marker='s')
    ax.annotate(name, (r['acc'], r['f1']),
                fontsize=8, xytext=(4,4), textcoords='offset points')
for name, r in {**hyb_results}.items():
    ax.scatter(r['acc'], r['f1'], c=TIER_COLORS['Hybrid'],
               s=180, zorder=5, marker='^')
    ax.annotate(name.replace('ResNet34 → ','CNN→'), (r['acc'], r['f1']),
                fontsize=8, xytext=(4,4), textcoords='offset points')
ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_ylabel('Macro-F1 (%)', fontsize=12)
ax.set_title('Accuracy vs F1 — all methods', fontweight='bold', fontsize=13)
ax.legend(handles=[Patch(color=v,label=k) for k,v in TIER_COLORS.items()],
          fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS/'acc_vs_f1_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save + download ───────────────────────────────────────────────
summary = {
    'classical_ml': {k: {kk:vv for kk,vv in v.items() if kk!='preds'}
                     for k,v in ml_results.items()},
    'deep_learning': {k: {kk:vv for kk,vv in v.items()
                          if kk not in ('preds','labels','history')}
                      for k,v in dl_results.items()},
    'hybrid': {k: {kk:vv for kk,vv in v.items() if kk!='preds'}
               for k,v in hyb_results.items()}
}
with open(RESULTS/'results.json','w') as f:
    json.dump(summary, f, indent=2)

!zip -r /content/paddy_study_results.zip /content/results/ -q
from google.colab import files
files.download('/content/paddy_study_results.zip')
print('All results downloaded.')

---
## Paper outputs from this notebook

| File | Paper section |
|---|---|
| `master_results.csv` | Table 1 — main comparison |
| `main_results.png` | Figure 2 — F1, Acc, Speed |
| `dl_training_curves.png` | Figure 3 — DL convergence |
| `confusion_matrix.png` | Figure 4 — per-class errors |
| `acc_vs_f1_scatter.png` | Figure 5 — Pareto frontier |

## Key findings to discuss in the paper

1. **Gap between ML and DL**: How large is it really? If <10% F1, that's the headline.
2. **Hybrid closes the gap**: Does CNN→SVM match end-to-end ResNet34? That's the contribution.
3. **Speed tradeoff**: Classical ML infers in microseconds vs DL milliseconds — on-farm deployment.
4. **Which diseases confuse each tier?**: Confusion matrices reveal whether each tier fails on the same hard classes.